# HW5: Building Inventory Visualizations
## Creating Interactive Plots with Altair

In [1]:
import pandas as pd
import numpy as np
import altair as alt

### Load Data from URL

In [2]:
# URL to the building inventory dataset
buildings_url = 'https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/building_inventory.csv'

In [3]:
buildings = pd.read_csv(buildings_url)
buildings.head()

,Agency Name,Location Name,Address,City,Zip code,County,Congress Dist,Congressional Full Name,Rep Dist,Rep Full Name,...,Bldg Status,Year Acquired,Year Constructed,Square Footage,Total Floors,Floors Above Grade,Floors Below Grade,Usage Description,Usage Description 2,Usage Description 3
0,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,1975,1975,144,1,1,0,Unusual,Unusual,Not provided
1,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided
2,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided
3,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided
4,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided


In [4]:
print(f"Dataset shape: {buildings.shape}")
print(f"\nColumns: {buildings.columns.tolist()}")

Dataset shape: (8862, 22)

Columns: ['Agency Name', 'Location Name', 'Address', 'City', 'Zip code', 'County', 'Congress Dist', 'Congressional Full Name', 'Rep Dist', 'Rep Full Name', 'Senate Dist', 'Senator Full Name', 'Bldg Status', 'Year Acquired', 'Year Constructed', 'Square Footage', 'Total Floors', 'Floors Above Grade', 'Floors Below Grade', 'Usage Description', 'Usage Description 2', 'Usage Description 3']


In [5]:
# Check data types and missing values
buildings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8862 entries, 0 to 8861
Data columns (total 22 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Agency Name              8862 non-null   object
 1   Location Name            8862 non-null   object
 2   Address                  8811 non-null   object
 3   City                     8862 non-null   object
 4   Zip code                 8862 non-null   int64 
 5   County                   8837 non-null   object
 6   Congress Dist            8862 non-null   int64 
 7   Congressional Full Name  8699 non-null   object
 8   Rep Dist                 8862 non-null   int64 
 9   Rep Full Name            8839 non-null   object
 10  Senate Dist              8862 non-null   int64 
 11  Senator Full Name        8839 non-null   object
 12  Bldg Status              8862 non-null   object
 13  Year Acquired            8862 non-null   int64 
 14  Year Constructed         8862 non-null  

### Data Transformation and Cleaning

In [6]:
# 1) filter out rows with missing square footage or year acquired
buildings_clean = buildings.dropna(subset=['Square Footage', 'Year Acquired'])

# 2) filter to keep only reasonable years (e.g., after 1800)
buildings_clean = buildings_clean[buildings_clean['Year Acquired'] > 1800]

# 3) filter to keep only positive square footage
buildings_clean = buildings_clean[buildings_clean['Square Footage'] > 0]

print(f"Cleaned dataset shape: {buildings_clean.shape}")

Cleaned dataset shape: (8572, 22)


## Plot 1: Distribution of Buildings by Agency
### Bar chart showing the top agencies by building count

In [7]:
# Get the top 15 agencies by building count
top_agencies = buildings['Agency Name'].value_counts().head(15).index.tolist()

# Create bar chart using URL data and filter in Altair
plot1 = alt.Chart(buildings_url).mark_bar().encode(
    x=alt.X('count():Q', title='Number of Buildings'),
    y=alt.Y('Agency Name:N', sort='-x', title='Agency'),
    color=alt.Color('count():Q', 
                    scale=alt.Scale(scheme='viridis'),
                    legend=alt.Legend(title='Building Count')),
    tooltip=['Agency Name:N', 'count():Q']
).transform_filter(
    alt.FieldOneOfPredicate(field='Agency Name', oneOf=top_agencies)
).properties(
    width=600,
    height=400,
    title='Top 15 Agencies by Number of Buildings'
)

plot1

alt.Chart(...)

## Plot 2: Interactive Scatter Plot with Brush Selection
### Year Acquired v.s. Square Footage with linked histogram

In [8]:
# Create brush selection
brush = alt.selection_interval(encodings=['x', 'y'])

# Scatter plot using URL data
scatter = alt.Chart(buildings_url).mark_circle(size=20, opacity=0.5).encode(
    x=alt.X('Year Acquired:Q', 
            scale=alt.Scale(domain=[1850, 2020]),
            title='Year Acquired'),
    y=alt.Y('Square Footage:Q', 
            scale=alt.Scale(type='log'),
            title='Square Footage (log scale)'),
    color=alt.condition(brush, 
                       alt.value('steelblue'), 
                       alt.value('lightgray')),
    tooltip=['Year Acquired:Q', 'Square Footage:Q', 'Agency Name:N']
).transform_filter(
    (alt.datum['Year Acquired'] > 1800) & (alt.datum['Square Footage'] > 0)
).properties(
    width=400,
    height=300,
    title='Building Acquisition Timeline'
).add_params(
    brush
)

# Histogram that responds to brush selection
histogram = alt.Chart(buildings_url).mark_bar().encode(
    x=alt.X('Year Acquired:Q', bin=alt.Bin(maxbins=30), title='Year Acquired'),
    y=alt.Y('count():Q', title='Number of Buildings'),
    color=alt.value('steelblue')
).transform_filter(
    (alt.datum['Year Acquired'] > 1800) & (alt.datum['Square Footage'] > 0)
).transform_filter(
    brush
).properties(
    width=400,
    height=200,
    title='Distribution of Selected Buildings by Year'
)

# Combine plots vertically
plot2 = alt.vconcat(scatter, histogram)

plot2

alt.VConcatChart(...)

### Save plots as JSON files:|

In [9]:
# 1) et the directory for saving JSON files
import os

# 2) Get the repo root directory (go up from python_notebooks to repo root)
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
json_dir = os.path.join(repo_root, 'assets', 'json')

# create directory if it doesn't exist
os.makedirs(json_dir, exist_ok=True)

print(f"Saving JSON files to: {json_dir}")

Saving JSON files to: /Users/shiweiliu/Documents/iSchool/2025Fall/2025Fall_Coursework/doudou6688-sweet.github.io/assets/json


In [10]:
# save plot 1
plot1.save(os.path.join(json_dir, 'hw5_plot1_agencies.json'))
print("Plot 1 saved successfully!")

Plot 1 saved successfully!


In [11]:
# save plot 2
plot2.save(os.path.join(json_dir, 'hw5_plot2_interactive.json'))
print("Plot 2 saved successfully!")

Plot 2 saved successfully!


In [12]:
# check file sizes (optional), just do this for practice as I have learned from class
plot1_size = os.stat(os.path.join(json_dir, 'hw5_plot1_agencies.json')).st_size
plot2_size = os.stat(os.path.join(json_dir, 'hw5_plot2_interactive.json')).st_size

print(f"Plot 1 file size: {plot1_size / 1024:.2f} KB")
print(f"Plot 2 file size: {plot2_size / 1024:.2f} KB")

Plot 1 file size: 1.29 KB
Plot 2 file size: 1.65 KB
